# V2 — raw EEGNet → ZuCo sentiment

This independent notebook asks whether local raw-EEG structure contains alignment-specific sentiment information without using NeuroLM features or sentence text. It trains one locked compact EEGNet configuration and compares aligned EEG with a separately trained split-local shuffled control, a temporal-block-shuffle diagnostic, and a majority baseline.

Use a **GPU** Colab runtime and run every cell in order. Preprocessed subject packs, partial fold results, and final outputs persist in Google Drive. V1 files and paths are never overwritten.

In [ ]:
# 1) Fetch this codebase and validate it using Colab's existing scientific stack.
from pathlib import Path
import os, subprocess, sys
import numpy as np, pandas as pd, scipy, sklearn, torch

PROJECT_URL = "https://github.com/parmisbathayan/EEGTokenizer.git"
PROJECT_ROOT = Path("/content/EEGTokenizer")

def run(command, **kwargs):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, check=True, text=True, **kwargs)

if not PROJECT_ROOT.exists():
    run(["git", "clone", "--depth", "1", PROJECT_URL, str(PROJECT_ROOT)])
else:
    run(["git", "pull", "--ff-only"], cwd=PROJECT_ROOT)
os.chdir(PROJECT_ROOT / "neurolm")
run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-q"], cwd=Path.cwd())
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime → Change runtime type → GPU, then rerun")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0))
print("NumPy/SciPy/scikit-learn:", np.__version__, scipy.__version__, sklearn.__version__)
print("Project ready:", Path.cwd())

In [ ]:
# 2) Mount Drive and edit only these paths if your thesis layout differs.
from google.colab import drive
drive.mount("/content/drive")

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis")
DATA_ROOT = THESIS_ROOT / "Data"
CACHE_ROOT = THESIS_ROOT / "CachedArtifacts/eeg_tokenizer/neurolm"
RESULTS_ROOT = THESIS_ROOT / "Results/eeg_tokenizer/neurolm"
RAW_DIR = DATA_ROOT / "zuco_og_raw"
LABELS_CSV = DATA_ROOT / "zuco_sentiment_labels_task1_fixed.csv"
RAW_PACKS = CACHE_ROOT / "raw_eeg_packs_v2"
RESULTS_DIR = RESULTS_ROOT / "raw_eegnet_v2"

for path in (RAW_DIR, LABELS_CSV):
    if not path.exists():
        raise FileNotFoundError(f"Edit this cell; not found: {path}")
for path in (RAW_PACKS, RESULTS_DIR):
    path.mkdir(parents=True, exist_ok=True)
print("Raw EEG:", RAW_DIR)
print("V2 cache:", RAW_PACKS)
print("V2 results:", RESULTS_DIR)

In [ ]:
# 3) Inspect ZuCo, then build/resume one preprocessed raw-EEG pack per subject.
import json
from src.config import PreprocessConfig
from src.raw_cache import build_raw_subject_packs
from src.zuco_io import inspect_zuco

inspection = pd.DataFrame(inspect_zuco(RAW_DIR, LABELS_CSV))
inspection.to_csv(RESULTS_DIR / "data_inspection.csv", index=False)
display(inspection)
print("Subjects:", len(inspection))
print("Label-matched rows:", int(inspection.matched_labels.sum()))
print("Usable raw recordings:", int(inspection.usable_for_neurolm.sum()))

preprocess_config = PreprocessConfig()
cache_manifest = build_raw_subject_packs(
    raw_dir=RAW_DIR,
    labels_csv=LABELS_CSV,
    pack_dir=RAW_PACKS,
    preprocess_config=preprocess_config,
    overwrite=False,
    progress_every=50,
)
print(json.dumps(cache_manifest["report"] | {"failure_rows": len(cache_manifest["report"]["failures"])}, indent=2))

In [ ]:
# 4) Load the compact packs once, save diagnostics, and smoke-test the locked model.
# Reload after Cell 1 pulls fixes into an already-running Colab kernel.
import importlib
import src.raw_cache as raw_cache
importlib.reload(raw_cache)
from src.raw_eegnet import RawEEGNetConfig, smoke_test_raw_eegnet

records, recording_metadata, dataset_report = raw_cache.load_raw_records(
    RAW_PACKS, preprocess_config
)
recording_metadata.to_csv(RESULTS_DIR / "recording_metadata.csv", index=False)
(RESULTS_DIR / "dataset_diagnostics.json").write_text(json.dumps(dataset_report, indent=2))
environment = {
    "python": sys.version,
    "torch": torch.__version__,
    "cuda_device": torch.cuda.get_device_name(0),
}
(RESULTS_DIR / "environment.json").write_text(json.dumps(environment, indent=2))
print(json.dumps(dataset_report, indent=2))

evaluation_config = RawEEGNetConfig()
smoke = smoke_test_raw_eegnet(records, evaluation_config, device="cuda")
print(json.dumps(smoke, indent=2))

In [ ]:
# 5) Run/resume the locked 3-seed × 5-fold evaluation and save the stoplight decision.
import matplotlib.pyplot as plt
from src.raw_eegnet import evaluate_raw_eegnet

metrics, predictions, summary, delta, gate = evaluate_raw_eegnet(
    records=records,
    output_dir=RESULTS_DIR,
    dataset_fingerprint=dataset_report["dataset_fingerprint"],
    config=evaluation_config,
    device="cuda",
)
display(summary)
print(json.dumps(gate, indent=2))

plot_rows = metrics.groupby("setup").macro_f1.agg(["mean", "std"]).sort_values("mean")
ax = plot_rows["mean"].plot.barh(xerr=plot_rows["std"], figsize=(8, 4), capsize=3)
ax.axvline(1 / 3, color="black", linestyle="--", linewidth=1, label="balanced chance")
ax.set_xlabel("Macro-F1 across folds")
ax.set_ylabel("")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "macro_f1_comparison.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved all V2 results:", RESULTS_DIR)

## Interpretation rule

Only a **green** result is eligible for later tuning, and no version is tuned until the full three-version screen is complete. A **yellow** result is recorded as suggestive without modification. A **red** result ends this V2 branch. The temporal-block-shuffle score is diagnostic; the primary gate always compares aligned EEG with the separately trained shuffled-pairing control.